In [ ]:

# CONFIGURAÇÕES INICIAIS DAS ANÁLISES (PRESENTES EM TODOS OS SCRIPTS)
# IMPORTAR BIBLIOTECAS ---
import sys
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CAMINHOS ---
SCRIPT_DIR = Path(__file__).resolve().parent        # caminho desse script
ANALYTICS_DIR = SCRIPT_DIR.parent                   # pasta desse script

# PARA IMPORTAR FUNÇÕES DE EXTRAIR CSV ---
if str(ANALYTICS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYTICS_DIR))
from utils.export_utils import exportar_csv


# ROOT DO PROJETO ---
PROJECT_ROOT = Path(__file__).resolve().parents[3]

# ROOT DOS OUTPUTS ---
OUTPUT_DIR = (PROJECT_ROOT/ "scripts"/ "Analytics"/ "outputs"/ "gold_01")
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# SPARK ---
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 01 - Como está estruturado o mercado brasileiro de Dados? ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# CAMINHO DO ARQUIVO ---
caminho_gold_01 = (PROJECT_ROOT/ "Gold"/ "perguntas_negocio"/ "gold_01_estrutura_mercado")

# BUSCA CSVs GERADOS PELO SPARK NA CRIAÇÃO DA GOLD ---
arquivos_gold_01 = [
    str(arquivo)
    for arquivo in caminho_gold_01.glob("part-*.csv")
]

print("Arquivos encontrados:")
print(arquivos_gold_01)

# CARREGAR GOLD 01 ---
df_estrutura = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_01)
)
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# INÍCIO DAS ANÁLISES ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# REGIÃO ONDE MORA ---
# ANÁLISE INICIAL DA DIMENSÃO ---
df_regiao = (df_estrutura
    .filter(F.col("variavel") == "regiao_onde_mora")
    .orderBy("edicao",F.desc("pct_na_dimensao"))
)
df_regiao.show(100, truncate=False)
print("Qtd linhas:", df_regiao.count())

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A dimensão de região possui informações para as três edições analisadas,
totalizando 15 registros.

As cinco regiões brasileiras aparecem em todos os períodos:
Sudeste, Sul, Nordeste, Centro-oeste e Norte.

O número de respondentes varia entre as edições:

- 2023-2024: 5.169 respondentes
- 2024-2025: 5.075 respondentes
- 2025-2026: 3.370 respondentes

Por esse motivo, as comparações históricas serão realizadas
principalmente pela participação percentual de cada região, e não
pela contagem absoluta.

A presença das mesmas cinco categorias nas três edições permite
comparação histórica direta.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# SITUAÇÃO DE TRABALHO ---
# ANÁLISE INICIAL DA DIMENSÃO ---
df_situacao = (df_estrutura
    .filter(F.col("variavel") == "situacao_de_trabalho")
    .orderBy("edicao",F.desc("pct_na_dimensao"))
)
df_situacao.show(100, truncate=False)
print("Qtd linhas:", df_situacao.count())

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A dimensão de situação de trabalho está disponível para as edições
2024-2025 e 2025-2026, totalizando 26 registros.

As mesmas 13 categorias aparecem nos dois períodos, permitindo
comparação direta entre as duas edições.

O número de respondentes varia entre os períodos:

- 2024-2025: 5.217 respondentes
- 2025-2026: 3.495 respondentes

Empregado (CLT) apresenta ampla predominância nas duas edições,
representando 72,5% da amostra em 2024-2025 e 65,2% em 2025-2026.

Apesar de permanecer como a principal situação de trabalho, sua
participação é menor na edição mais recente.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# NÚMERO DE FUNCIONÁRIOS DA EMPRESA ---
# ANÁLISE INICIAL DA DIMENSÃO ---
df_porte = (df_estrutura
    .filter(F.col("variavel") == "numero_de_funcionarios")
    .orderBy("edicao",F.desc("pct_na_dimensao"))
)

df_porte.show(100, truncate=False)
print("Qtd linhas:", df_porte.count())

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A dimensão de número de funcionários representa o porte das empresas
em que trabalham os respondentes.

O número de respondentes varia entre as edições:

- 2023-2024: 4.753 respondentes
- 2024-2025: 4.863 respondentes
- 2025-2026: 3.228 respondentes

A categoria "Acima de 3.000" apresenta a maior participação nas três
edições.

Durante a inspeção inicial também foi identificada a categoria
"de 501 a 100", que não representa uma faixa numérica coerente e
precisa ser investigada antes da comparação histórica.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# CATEGORIAS EXISTENTES POR EDIÇÃO ---
for nome, df in [
    ("REGIÃO", df_regiao),
    ("SITUAÇÃO DE TRABALHO", df_situacao),
    ("PORTE DA EMPRESA", df_porte),
]:
    print(f"\n--- {nome} ---")
    (
        df
        .select("edicao", "valor")
        .distinct()
        .orderBy("edicao", "valor")
        .show(200, truncate=False)
    )

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A inspeção das categorias confirma diferentes condições de
comparabilidade entre as dimensões.

REGIÃO:
As mesmas cinco categorias aparecem nas três edições, permitindo
comparação histórica direta.

SITUAÇÃO DE TRABALHO:
As mesmas 13 categorias aparecem em 2024-2025 e 2025-2026.
Como a dimensão não está disponível em 2023-2024, a comparação será
realizada apenas entre as duas edições mais recentes.

PORTE DA EMPRESA:
As principais faixas permanecem consistentes entre as edições.

Entretanto, a categoria "de 501 a 100" aparece em 2023-2024 e
2025-2026 e apresenta uma faixa numérica inconsistente.
Por esse motivo, será investigada separadamente antes da análise
histórica de porte.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# COMPARATIVO HISTÓRICO POR REGIÃO ---
comparativo_regiao = (df_regiao
    .groupBy("valor")
    .pivot("edicao",["2023-2024", "2024-2025", "2025-2026"])
    .agg(F.first("pct_na_dimensao"))
    .withColumn("variacao_pp",F.round(F.col("2025-2026") - F.col("2023-2024"),2))
    .orderBy(F.desc("variacao_pp"))
)
comparativo_regiao.show(truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
O Sudeste apresenta ampla predominância na distribuição regional
dos respondentes durante todo o período analisado.

Sua participação passa de 61,4% em 2023-2024 para 64,4% em
2025-2026, representando aumento de +3,0 pontos percentuais.

O Centro-oeste apresenta pequena variação positiva, passando de
6,7% para 6,9% (+0,2 p.p.).

A maior redução ocorre na região Sul, cuja participação passa de
18,6% para 16,0%, uma diferença de -2,6 pontos percentuais.

Nordeste apresenta redução de -0,5 p.p., enquanto Norte permanece
relativamente estável, com variação de -0,2 p.p.

Os resultados mostram que a concentração dos respondentes no Sudeste
permanece elevada e aumenta na edição mais recente.

PONTO DE ATENÇÃO:
A distribuição representa a localização dos respondentes das pesquisas
e não deve ser interpretada isoladamente como distribuição de todos
os profissionais de Dados no Brasil.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# COMPARATIVO DA SITUAÇÃO DE TRABALHO ---
comparativo_situacao = (df_situacao
    .groupBy("valor")
    .pivot("edicao",["2024-2025", "2025-2026"])
    .agg(F.first("pct_na_dimensao"))
    .withColumn("variacao_pp",F.round(F.col("2025-2026") - F.col("2024-2025"),2))
    .orderBy(F.desc("variacao_pp"))
)
comparativo_situacao.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Empregado (CLT) permanece como a principal situação de trabalho,
mas apresenta a maior redução de participação entre as categorias
analisadas.

A participação de profissionais CLT passa de 72,5% em 2024-2025
para 65,2% em 2025-2026, uma diferença de -7,3 pontos percentuais.

No mesmo período, algumas categorias aumentam sua participação:

- Empreendedor ou Empregado (CNPJ): +2,7 p.p.
- Vivo no Brasil e trabalho remoto para empresa de fora: +1,5 p.p.
- Servidor Público: +1,1 p.p.
- Desempregado, buscando recolocação: +0,9 p.p.

A participação de Estagiários permanece estável em 3,6%.

Os resultados indicam menor concentração da amostra na categoria
CLT na edição mais recente, acompanhada pelo aumento relativo de
outras situações profissionais.

PONTO DE ATENÇÃO:
As variações representam mudanças na composição dos respondentes
e não permitem concluir isoladamente que houve redução equivalente
do emprego CLT em todo o mercado de trabalho brasileiro.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# INVESTIGAÇÃO DE CATEGORIAS SUSPEITAS DE PORTE ---
(df_porte
    .filter(F.col("valor") == "de 501 a 100")
    .select("edicao","valor","contagem","pct_na_dimensao")
    .show(truncate=False)
)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
A investigação da categoria "de 501 a 100" confirma apenas um
registro em 2023-2024 e um registro em 2025-2026.

Em ambos os períodos, sua participação percentual é de aproximadamente
0,0% devido à baixa representatividade na amostra.

Além da frequência mínima, a descrição da faixa é numericamente
inconsistente, pois o limite inicial é superior ao limite final.

Por esse motivo, a categoria será desconsiderada apenas nas análises
de porte deste script.

A exclusão ocorre somente na base analítica e não altera os dados
originais da camada Gold.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# REMOVE CATEGORIA INCONSISTENTE APENAS DA ANÁLISE DE PORTE ---
df_porte_analise = (df_porte
    .filter(F.col("valor") != "de 501 a 100")
)


# COMPARATIVO HISTÓRICO DO PORTE DAS EMPRESAS ---
comparativo_porte = (df_porte_analise
    .groupBy("valor")
    .pivot("edicao",["2023-2024", "2024-2025", "2025-2026"])
    .agg(F.first("pct_na_dimensao"))
    .withColumn("variacao_pp",F.round(F.col("2025-2026") - F.col("2023-2024"),2))
    .orderBy(F.desc("variacao_pp"))
)
comparativo_porte.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Empresas com mais de 3.000 funcionários permanecem como a categoria
de porte com maior participação nas três edições.

Sua participação passa de 43,7% em 2023-2024 para 48,0% em
2024-2025 e recua para 40,9% em 2025-2026.

Na comparação entre a primeira e a última edição, isso representa
redução de -2,8 pontos percentuais.

Entre as faixas com aumento de participação destacam-se:

- de 51 a 100 funcionários: +1,5 p.p.
- de 501 a 1.000 funcionários: +1,2 p.p.
- de 1.001 a 3.000 funcionários: +0,8 p.p.
- de 1 a 5 funcionários: +0,3 p.p.

A faixa de 101 a 500 funcionários apresenta redução de -0,6 p.p.

Apesar das variações, organizações com mais de 3.000 funcionários
continuam apresentando participação significativamente superior às
demais categorias de porte.

PONTO DE ATENÇÃO:
Os percentuais representam o porte das empresas dos respondentes
da pesquisa e não a distribuição de todas as empresas que empregam
profissionais de Dados no mercado brasileiro.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# ESTRUTURA ATUAL POR PORTE - 2025-2026
porte_atual = (df_porte_analise
    .filter(F.col("edicao") == "2025-2026")
    .select("valor","contagem","pct_na_dimensao")
    .orderBy(F.desc("pct_na_dimensao"))
)
porte_atual.show(100, truncate=False)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
"""
Na edição mais recente, empresas com mais de 3.000 funcionários
representam a principal categoria de porte da amostra, concentrando
40,9% dos respondentes.

Na sequência aparecem:

- de 101 a 500 funcionários: 16,7%
- de 1.001 a 3.000 funcionários: 11,7%
- de 501 a 1.000 funcionários: 10,6%
- de 51 a 100 funcionários: 8,4%
- de 11 a 50 funcionários: 7,2%
- de 1 a 5 funcionários: 2,9%
- de 6 a 10 funcionários: 1,6%

A distribuição mostra forte presença de profissionais vinculados
a empresas de grande porte dentro da amostra, especialmente
organizações com mais de 3.000 funcionários.
"""
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------


# -------------------------------------------------------------------------------------------------------------------------------------------------------------------
# EXPORTAÇÃO DOS RESULTADOS PARA VISUALIZAÇÃO ---
# -------------------------------------------------------------------------------------------------------------------------------------------------------------------

# REGIAO HISTORICO.CSV
exportar_csv(comparativo_regiao,OUTPUT_DIR,"regiao_historico.csv")

# SITUAÇÃO TRABALHO HISTORICO.CSV
exportar_csv(comparativo_situacao,OUTPUT_DIR,"situacao_trabalho_historico.csv")

# PORTE ATUAL.CSV
exportar_csv(porte_atual,OUTPUT_DIR,"porte_atual.csv")

# PORTE HISTORICO.CSV
exportar_csv(comparativo_porte,OUTPUT_DIR,"porte_historico.csv")